# Решения: Проект 18 — рынок труда IT в США

Решения упражнений из `18_IT_Job_Market_USA.ipynb`. Ноутбук самодостаточен:
данные (снимок BLS/Census/Tax Foundation) и базовая модель воспроизведены ниже.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# --- снимок проекта 18 (BLS OEWS 2023 / Census ACS 2023 / Tax Foundation 2024) ---
SNAPSHOT = {
  'AL': ('Alabama',104000,950,179000,62000,0.045), 'AK': ('Alaska',112000,1320,318000,89000,0.0),
  'AZ': ('Arizona',119000,1340,340000,74000,0.025), 'AR': ('Arkansas',96000,870,167000,58000,0.039),
  'CA': ('California',159000,1900,725000,96000,0.093), 'CO': ('Colorado',130000,1600,480000,92000,0.044),
  'CT': ('Connecticut',124000,1340,343000,91000,0.055), 'DE': ('Delaware',116000,1310,330000,82000,0.055),
  'DC': ('District of Columbia',138000,1800,705000,108000,0.085), 'FL': ('Florida',112000,1550,348000,71000,0.0),
  'GA': ('Georgia',122000,1300,320000,75000,0.055), 'HI': ('Hawaii',115000,1900,840000,98000,0.079),
  'ID': ('Idaho',105000,1150,430000,74000,0.058), 'IL': ('Illinois',121000,1200,265000,81000,0.0495),
  'IN': ('Indiana',100000,980,200000,71000,0.0315), 'IA': ('Iowa',99000,900,199000,74000,0.057),
  'KS': ('Kansas',104000,970,210000,73000,0.057), 'KY': ('Kentucky',98000,900,192000,66000,0.045),
  'LA': ('Louisiana',101000,1010,200000,62000,0.0425), 'ME': ('Maine',104000,1050,300000,74000,0.0715),
  'MD': ('Maryland',130000,1560,405000,100000,0.0575), 'MA': ('Massachusetts',140000,1700,560000,101000,0.05),
  'MI': ('Michigan',104000,1050,224000,71000,0.0425), 'MN': ('Minnesota',116000,1230,320000,87000,0.0785),
  'MS': ('Mississippi',92000,900,155000,55000,0.047), 'MO': ('Missouri',106000,970,220000,70000,0.0475),
  'MT': ('Montana',100000,1000,350000,71000,0.059), 'NE': ('Nebraska',104000,970,230000,74000,0.0584),
  'NV': ('Nevada',108000,1450,415000,76000,0.0), 'NH': ('New Hampshire',120000,1400,400000,96000,0.0),
  'NJ': ('New Jersey',134000,1600,430000,101000,0.0637), 'NM': ('New Mexico',108000,1000,265000,62000,0.049),
  'NY': ('New York',137000,1550,420000,84000,0.0685), 'NC': ('North Carolina',120000,1200,300000,72000,0.045),
  'ND': ('North Dakota',98000,920,260000,76000,0.0225), 'OH': ('Ohio',107000,960,205000,70000,0.035),
  'OK': ('Oklahoma',100000,920,185000,63000,0.0475), 'OR': ('Oregon',126000,1450,480000,80000,0.09),
  'PA': ('Pennsylvania',112000,1150,250000,76000,0.0307), 'RI': ('Rhode Island',118000,1250,380000,84000,0.0499),
  'SC': ('South Carolina',108000,1150,270000,68000,0.064), 'SD': ('South Dakota',97000,870,245000,71000,0.0),
  'TN': ('Tennessee',116000,1200,300000,68000,0.0), 'TX': ('Texas',130000,1350,300000,76000,0.0),
  'UT': ('Utah',118000,1350,480000,89000,0.0465), 'VT': ('Vermont',104000,1150,300000,78000,0.066),
  'VA': ('Virginia',133000,1450,375000,91000,0.0575), 'WA': ('Washington',158000,1650,560000,96000,0.0),
  'WV': ('West Virginia',95000,800,155000,58000,0.051), 'WI': ('Wisconsin',104000,1000,250000,75000,0.053),
  'WY': ('Wyoming',99000,900,320000,74000,0.0),
}
df = pd.DataFrame([(a, v[0], v[1], v[2], v[3], v[4], v[5]) for a, v in SNAPSHOT.items()],
                  columns=['abbr','state','dev_wage','rent','home_value','hh_income','state_tax'])

FED_BRACKETS = [(0,11600,0.10),(11600,47150,0.12),(47150,100525,0.22),(100525,191950,0.24),
                (191950,243725,0.32),(243725,609350,0.35),(609350,np.inf,0.37)]
STD_DEDUCTION = 14600
def federal_tax(gross):
    taxable = max(0.0, gross - STD_DEDUCTION); tax = 0.0
    for lo, hi, rate in FED_BRACKETS:
        if taxable > lo: tax += (min(taxable, hi) - lo) * rate
        else: break
    return tax

MORTGAGE_RATE, DOWN_PAYMENT, YEARS = 0.07, 0.20, 30
def annual_mortgage(home_value, rate=MORTGAGE_RATE):
    P = home_value * (1 - DOWN_PAYMENT); r = rate / 12; n = YEARS * 12
    return P * r * (1 + r) ** n / ((1 + r) ** n - 1) * 12

def build(df, wage_col='dev_wage'):
    df = df.copy()
    df['fed_tax'] = df[wage_col].apply(federal_tax)
    df['net_salary'] = df[wage_col] - df['fed_tax'] - df[wage_col] * df['state_tax']
    df['annual_rent'] = df['rent'] * 12
    df['annual_mortgage'] = df['home_value'].apply(annual_mortgage)
    df['dispo_rent'] = df['net_salary'] - df['annual_rent']
    df['dispo_buy'] = df['net_salary'] - df['annual_mortgage']
    df['price_to_income'] = df['home_value'] / df[wage_col]
    return df

def zscore(s): return (s - s.mean()) / s.std(ddof=0)
WEIGHTS = {'net_salary':0.35,'dispo_rent':0.30,'annual_rent':-0.15,'state_tax':-0.10,'price_to_income':-0.10}
def compute_score(d, w):
    s = pd.Series(0.0, index=d.index)
    for col, wt in w.items(): s += wt * zscore(d[col])
    return s

df = build(df)
df['score'] = compute_score(df, WEIGHTS)
print('Топ-5 (базовый рейтинг):')
print(df.sort_values('score', ascending=False)[['state','net_salary','dispo_rent']].head())

## Упражнение 1: Полный cost-of-living (RPP + налог с продаж)

Скорректируем располагаемый доход на **уровень цен** и **налог с продаж**. Здесь
используем прокси индекса цен на базе аренды (RPP-прокси, США=100) и приблизительные
комбинированные ставки налога с продаж (штат+местный, 2024). Реальный располагаемый
доход = (доход − аренда − налог с продаж на потребление) / индекс_цен.

In [ ]:
# Комбинированный налог с продаж (штат+местный, приблизительно 2024); по умолчанию 7%
SALES_TAX = {'CA':0.0885,'TX':0.0820,'WA':0.0940,'NY':0.0852,'FL':0.0700,'TN':0.0955,
             'OR':0.0,'MT':0.0,'NH':0.0,'DE':0.0,'CO':0.0777,'AZ':0.0840,'IL':0.0880,
             'GA':0.0740,'NV':0.0823,'MA':0.0625,'PA':0.0634,'VA':0.0575,'NC':0.0698,
             'AK':0.0176,'WY':0.0536,'CQ':0.0}
df2 = df.copy()
df2['sales_tax'] = df2['abbr'].map(SALES_TAX).fillna(0.07)

# RPP-прокси: относительный уровень цен на базе аренды (США = 100)
df2['rpp'] = 100 * df2['rent'] / df2['rent'].mean()

# Потребление ~ 45% дохода на руки облагается налогом с продаж
consumption = 0.45 * df2['net_salary']
df2['real_dispo'] = (df2['dispo_rent'] - df2['sales_tax'] * consumption) / (df2['rpp'] / 100)

top = df2.sort_values('real_dispo', ascending=False)[
    ['state','dispo_rent','sales_tax','rpp','real_dispo']].head(10).copy()
top['dispo_rent'] = top['dispo_rent'].map('${:,.0f}'.format)
top['sales_tax'] = (top['sales_tax']*100).map('{:.1f}%'.format)
top['rpp'] = top['rpp'].map('{:.0f}'.format)
top['real_dispo'] = top['real_dispo'].map('${:,.0f}'.format)
print('Топ-10 по РЕАЛЬНОМУ располагаемому доходу (с поправкой на цены и налог с продаж):')
print(top.to_string(index=False))
print('\nВывод: штаты с низкой арендой/ценами (TX, TN, штаты без налога с продаж)')
print('поднимаются; дорогие побережья (CA, NY) опускаются.')

## Упражнение 2: Другая роль (Data Scientist / DevOps)

Возьмём зарплаты Data Scientist (≈ ×1.08 к разработчику) и DevOps (≈ ×1.03).
Из-за прогрессивного федерального налога рост зарплаты немного меняет и рейтинг.

In [ ]:
for role, mult in [('Data Scientist', 1.08), ('DevOps', 1.03)]:
    d = df[['abbr','state','rent','home_value','hh_income','state_tax']].copy()
    d['dev_wage'] = df['dev_wage'] * mult
    d = build(d)
    d['score'] = compute_score(d, WEIGHTS)
    top5 = d.sort_values('score', ascending=False)['state'].head(5).tolist()
    print(f'{role} (×{mult}): топ-5 -> {top5}')
print('\nЛидеры почти те же (WA/TX/штаты без налога), т.к. z-score устойчив к')
print('масштабированию; прогрессия налога даёт лишь небольшие сдвиги.')

## Упражнение 3: Покупка vs аренда

Рейтинг по `dispo_buy` (ипотека 7%) и поиск ставки, при которой покупка выгоднее аренды
(годовой платёж по ипотеке = годовой аренде) для лучшего штата.

In [ ]:
db = df.copy()
tmp = df.copy(); tmp['dispo_rent'] = tmp['dispo_buy']
db['score_buy'] = compute_score(tmp, WEIGHTS)
top_buy = db.sort_values('dispo_buy', ascending=False)[['state','dispo_rent','dispo_buy']].head(8).copy()
for col in ['dispo_rent','dispo_buy']:
    top_buy[col] = top_buy[col].map('${:,.0f}'.format)
print('Топ-8 по располагаемому доходу при ПОКУПКЕ (ипотека 7%):')
print(top_buy.to_string(index=False))

# Ставка безубыточности: annual_mortgage(rate) == annual_rent для штата-лидера
row = df.sort_values('dispo_rent', ascending=False).iloc[0]
def gap(rate): return annual_mortgage(row['home_value'], rate) - row['annual_rent']
lo, hi = 0.001, 0.20
for _ in range(60):
    mid = (lo + hi) / 2
    if gap(mid) > 0: hi = mid
    else: lo = mid
print(f'\n{row["state"]}: покупка (50м²-эквивалент дома) выгоднее аренды при ставке < {mid*100:.2f}%')
print(f'(при текущих 7% годовая ипотека ${annual_mortgage(row["home_value"]):,.0f} vs аренда ${row["annual_rent"]:,.0f})')

## Упражнение 4: Свежие данные через Census API

Живая загрузка из бесплатного Census ACS API (нужен интернет; ключ не обязателен для
небольших запросов). Код защищён `try/except` и корректно откатывается офлайн.

In [ ]:
import json, urllib.request
def fetch_census(year=2023):
    url = (f'https://api.census.gov/data/{year}/acs/acs1'
           '?get=NAME,B25064_001E,B25077_001E,B19013_001E&for=state:*')
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=15) as r:
        raw = json.load(r)
    hdr, rows = raw[0], raw[1:]
    out = pd.DataFrame(rows, columns=hdr)
    return out[['NAME','B25064_001E','B25077_001E']].head()

try:
    live = fetch_census(2023)
    print('Живые данные Census ACS 2023 (медианная аренда/стоимость жилья):')
    print(live.to_string(index=False))
except Exception as e:
    print(f'Census API недоступен ({type(e).__name__}); используется встроенный снимок.')
    print('При наличии интернета функция вернёт свежие данные ACS по штатам/метро.')

## Упражнение 5: Свои приоритеты (налог важнее всего)

In [ ]:
MY_WEIGHTS = {'net_salary':0.25,'dispo_rent':0.20,'annual_rent':-0.10,
              'state_tax':-0.35,'price_to_income':-0.10}
d = df.copy(); d['my_score'] = compute_score(d, MY_WEIGHTS)
top5 = d.sort_values('my_score', ascending=False)[['state','state_tax','net_salary','dispo_rent']].head(5).copy()
top5['state_tax'] = (top5['state_tax']*100).map('{:.1f}%'.format)
for col in ['net_salary','dispo_rent']: top5[col] = top5[col].map('${:,.0f}'.format)
print('Мой топ-5 (приоритет — низкий налог штата):')
print(top5.to_string(index=False))
print('\nВперёд выходят штаты без подоходного налога: WA, TX, FL, TN, NV.')